# 5. TUR — GGUF'a çevir

**Bu defteri TEMİZ bir runtime'da aç.** Eğitim yaptığın oturumda değil.

Colab'da: `Runtime → Disconnect and delete runtime` de, sonra bu defteri
yükleyip baştan sona çalıştır. Yeniden başlatma gerekmiyor, tek geçiş.

---

### ⚠️ Neden temiz runtime

llama.cpp'nin `requirements-convert_hf_to_gguf.txt` dosyası **numpy'ı 1.26'ya
düşürüyor**, oysa Colab'ın torch'u numpy 2.x'e göre derlenmiş. Sonuç:

```
ValueError: numpy.dtype size changed, may indicate binary incompatibility.
```

Bu defter o requirements dosyasını **kurmuyor** — `gguf` modülü zaten
llama.cpp deposunun içinde (`gguf-py/`) geliyor ve convert betiği onu
kendisi buluyor. numpy'a hiç dokunulmuyor.

| adım | süre |
|---|---|
| 1. temizlik + kurulum | ~2 dk |
| 2. llama.cpp | ~1 dk |
| 3. LoRA'yı birleştir | ~5 dk |
| 4. GGUF'a çevir | ~5 dk |
| 5. Drive'a kopyala | ~1 dk |

## 1. Ortam

⚠️ `numpy`, `torch`, `transformers` sürümlerine DOKUNULMUYOR.

In [ ]:
!pip uninstall -y -q tensorflow tensorflow-cpu tensorboard tensorflow-metadata torchao 2>/dev/null
!pip install -q peft sentencepiece

import numpy, torch
print('numpy :', numpy.__version__, '  <- 2.x olmali')
print('torch :', torch.__version__)


## 2. Drive + llama.cpp

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone --depth 1 -q https://github.com/ggerganov/llama.cpp /content/llama.cpp
print('llama.cpp hazir')


## 3. Adaptörü bul

In [ ]:
from pathlib import Path
from datetime import datetime

CIKTI = Path('/content/drive/MyDrive/codifya/cikti')
TUR_ETIKETI = 'tur6'
LORA_YOLU = CIKTI / f'b33_{TUR_ETIKETI}_lora'

for d in sorted(CIKTI.iterdir()):
    a = d / 'adapter_model.safetensors' if d.is_dir() else None
    if a is not None and a.exists():
        t = datetime.fromtimestamp(a.stat().st_mtime)
        print(f'  {d.name:24s} {a.stat().st_size / 1024**2:7.1f} MB   {t:%Y-%m-%d %H:%M}')

assert (LORA_YOLU / 'adapter_model.safetensors').exists(), f'adapter yok: {LORA_YOLU}'
print()
print('kullanilacak:', LORA_YOLU)


## 4. LoRA'yı taban modelle birleştir

CPU'da yapılıyor, GPU gerekmiyor. ~5 dakika.

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

TABAN = 'Qwen/Qwen2.5-1.5B-Instruct'
BIRLESIK = Path('/content/birlesik')

taban = AutoModelForCausalLM.from_pretrained(TABAN, torch_dtype=torch.float16, device_map='cpu')
tokenizer = AutoTokenizer.from_pretrained(TABAN)

model = PeftModel.from_pretrained(taban, str(LORA_YOLU)).merge_and_unload()
model.save_pretrained(str(BIRLESIK), safe_serialization=True)
tokenizer.save_pretrained(str(BIRLESIK))

gb = sum(f.stat().st_size for f in BIRLESIK.rglob('*') if f.is_file()) / 1024**3
print(f'birlestirildi: {BIRLESIK}  ({gb:.2f} GB)')


## 5. GGUF'a çevir (q8_0)

In [ ]:
import os

GGUF = f'/content/codifya-{TUR_ETIKETI}-q8_0.gguf'

!python /content/llama.cpp/convert_hf_to_gguf.py /content/birlesik --outfile {GGUF} --outtype q8_0

print()
print('GGUF :', GGUF)
print('boyut:', round(os.path.getsize(GGUF) / 1024**3, 2), 'GB')


## 6. Drive'a kopyala

In [ ]:
import shutil

hedef = CIKTI / os.path.basename(GGUF)
shutil.copy(GGUF, hedef)

print('=' * 62)
print('BITTI')
print('=' * 62)
print('Dosya:', hedef)
print('Boyut:', round(hedef.stat().st_size / 1024**3, 2), 'GB')
print()
print('SIRADA: Drive > codifya > cikti klasorunden bu dosyayi indir ve')
print('        D:/ERP/codifya-decision-engine/training/ klasorune koy.')
